# Solution: Stable Diffusion Foundations — Toy DDPM

This notebook trains a tiny DDPM on a two-dimensional target distribution. The model learns to predict the noise added at a randomly selected diffusion step, then generates new points by reversing the noise process.

> **Notation:** `B` = batch size, `D` = data dimension (`2` here), and `T` = number of diffusion steps (`100` here).


## 1. Forward diffusion and the training target

For a clean point $x_0$, a schedule $\beta_t$ defines $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s=1}^{t}\alpha_s$. We can sample a noisy point directly, without simulating every earlier step:

$$x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1 - \bar\alpha_t}\epsilon, \qquad \epsilon \sim \mathcal N(0, I).$$

The network receives `x_t` and a normalized timestep, then predicts the exact noise $\epsilon$. Its MSE is the DDPM training loss.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim


def sample_p1(n, device=None):
    """Sample n points from the 2D target distribution.

    Returns:
        Tensor of shape (n, 2).
    """
    angle = torch.rand(n, device=device) * 2 * torch.pi  # (n,)
    radius = 2 + 0.1 * torch.randn(n, device=device)    # (n,)
    return torch.stack(
        (radius * torch.cos(angle), radius * torch.sin(angle) * torch.cos(angle)),
        dim=1,
    )  # (n, 2)


def make_noise_schedule(n_steps, device):
    """Create beta, alpha, and cumulative-alpha tensors, each shape (T,)."""
    betas = torch.linspace(1e-4, 2e-2, n_steps, device=device)  # (T,)
    alphas = 1 - betas                                           # (T,)
    alpha_bars = torch.cumprod(alphas, dim=0)                    # (T,)
    return betas, alphas, alpha_bars


class NoisePredictor(nn.Module):
    """An MLP that maps a 2D noisy point and time to predicted 2D noise."""

    def __init__(self, hidden_dim=128):
        super().__init__()
        # Input: [x_t[:, 0], x_t[:, 1], normalized_t] -> shape (B, 3)
        # Output: predicted noise epsilon_theta -> shape (B, 2)
        self.net = nn.Sequential(
            nn.Linear(3, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x, t):
        # x: (B, 2); t: (B,) or (B, 1) -> concatenate to (B, 3)
        if t.ndim == 1:
            t = t[:, None]
        return self.net(torch.cat((x, t), dim=1))  # (B, 2)


In [ ]:
def diffusion_loss(model, x0, alpha_bars):
    """Return the noise-prediction MSE for one minibatch.

    Args:
        x0: clean samples, shape (B, D).
        alpha_bars: cumulative schedule values, shape (T,).
    """
    batch_size, n_steps = x0.shape[0], alpha_bars.shape[0]

    # One independent integer timestep per training example: shape (B,)
    t = torch.randint(n_steps, (batch_size,), device=x0.device)
    # Advanced indexing selects one alpha_bar per item; (B, 1) broadcasts over D.
    alpha_bar_t = alpha_bars[t, None]  # (B, 1)

    noise = torch.randn_like(x0)  # epsilon, shape (B, D)
    xt = alpha_bar_t.sqrt() * x0 + (1 - alpha_bar_t).sqrt() * noise  # (B, D)

    normalized_t = t.to(x0.dtype)[:, None] / (n_steps - 1)  # (B, 1), in [0, 1]
    predicted_noise = model(xt, normalized_t)                 # (B, D)
    return ((predicted_noise - noise) ** 2).mean()


## 2. Reverse DDPM sampling

Starting from Gaussian noise $x_T$, predict $\epsilon_\theta(x_t, t)$, reconstruct a clean-point estimate $\hat{x}_0$, and draw $x_{t-1}$ from the DDPM posterior. At the final step (`t = 0`) we return its mean without adding noise.

$$\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar\alpha_t}\,\epsilon_\theta(x_t, t)}{\sqrt{\bar\alpha_t}}.$$

In [ ]:
@torch.no_grad()
def sample_ddpm(model, x_T, betas, alphas, alpha_bars):
    """Run the reverse process from x_T to a generated x_0.

    x_T has shape (B, D); schedule tensors have shape (T,).
    The returned generated samples also have shape (B, D).
    """
    was_training = model.training
    model.eval()
    x = x_T.clone()  # current state x_t, shape (B, D)
    n_steps = len(betas)

    for step in range(n_steps - 1, -1, -1):
        # The network expects time in [0, 1], repeated for every batch element.
        t = torch.full(
            (x.shape[0], 1), step / (n_steps - 1), device=x.device, dtype=x.dtype
        )  # (B, 1)
        predicted_noise = model(x, t)  # epsilon_theta(x_t, t), shape (B, D)

        alpha_t = alphas[step]          # scalar
        alpha_bar_t = alpha_bars[step]  # scalar
        alpha_bar_prev = (
            torch.ones_like(alpha_bar_t) if step == 0 else alpha_bars[step - 1]
        )  # scalar; alpha_bar_(-1) is defined as 1

        x0_estimate = (
            x - (1 - alpha_bar_t).sqrt() * predicted_noise
        ) / alpha_bar_t.sqrt()  # (B, D)

        # Posterior q(x_{t-1} | x_t, x_0) mean, shape (B, D).
        mean = (
            alpha_bar_prev.sqrt() * betas[step] / (1 - alpha_bar_t) * x0_estimate
            + alpha_t.sqrt() * (1 - alpha_bar_prev) / (1 - alpha_bar_t) * x
        )

        if step == 0:
            x = mean
        else:
            posterior_std = (
                betas[step] * (1 - alpha_bar_prev) / (1 - alpha_bar_t)
            ).sqrt()  # scalar
            x = mean + posterior_std * torch.randn_like(x)

    model.train(was_training)
    return x


## 3. Train and compare samples

`sample_p1(512)` produces a fresh training batch each update. After training, we compare 1,000 generated points with 1,000 independently sampled target points.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_steps = 100

betas, alphas, alpha_bars = make_noise_schedule(n_steps, device)
model = NoisePredictor().to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for step in range(1, 4_001):
    x0 = sample_p1(512, device)  # (B=512, D=2)
    loss = diffusion_loss(model, x0, alpha_bars)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 800 == 0:
        print(f'step {step:>4d} | loss: {loss.item():.4f}')

generated = sample_ddpm(
    model, torch.randn(1_000, 2, device=device), betas, alphas, alpha_bars
).cpu()  # (1_000, 2)
target = sample_p1(1_000).cpu()  # (1_000, 2)

plt.figure(figsize=(7, 5))
plt.scatter(target[:, 0], target[:, 1], s=4, alpha=0.5, label='target')
plt.scatter(generated[:, 0], generated[:, 1], s=4, alpha=0.5, label='generated')
plt.axis('equal')
plt.legend()
plt.show()


In [ ]:
from torch_judge import check

check('stable_diffusion')
